<a href="https://colab.research.google.com/github/AngelTroncoso/Alergias/blob/main/Pipeline_Bioinform%C3%A1tico_en_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Justificación Técnica para la Hackaton

## 1. Aceleración de la Identificación (Fase I)
El uso de APIs de bases de datos como **IEDB** ahorra meses de ensayos de unión *in vitro* al predecir qué péptidos de *Der p 1* o *Der p 23* tienen mayor afinidad por MHC II.

## 2. Validación de la Cura (Fase II)
El diseño de sgRNA dirigido no solo al alérgeno, sino a la maquinaria de supervivencia de las células plasmáticas de vida larga (LLPC) —como el gen **BCL2** mencionado en el reporte— abre la puerta a eliminar la "memoria" alérgica de forma definitiva.

## 3. Medición de Avances (Fases III y IV)
El pipeline permite monitorizar el éxito mediante el biomarcador $\Delta_{15w}$, permitiendo interrumpir protocolos ineficaces de forma temprana y personalizar la dosis según el fenotipo del paciente.

## 4. Viabilidad en Colab
Todas las librerías seleccionadas (`BioPython`, `XGBoost`, `SHAP`) son compatibles con el entorno gratuito de Google, facilitando la colaboración rápida durante el evento.

---

> Este pipeline proporciona la infraestructura lógica para transformar los datos moleculares y clínicos del reporte en una herramienta de decisión en tiempo real.

# 1. Configuración del Entorno y Librerías  
Este módulo instala las herramientas necesarias para el análisis de secuencias, modelado de Machine Learning y explicabilidad.

In [7]:
# Módulo 0: Instalación de dependencias críticas
%%capture
!pip install biopython xgboost shap pandas numpy requests
!pip install FlowKit # Para análisis de citometría si hay datos de activación de basófilos

# 2. Módulo de Inmunoinformática: Mapeo de Epítopos de Precisión  
Basado en la Fase I del documento, este bloque permite consultar el IEDB (Immune Epitope Database) para identificar péptidos candidatos de Dermatophagoides que interactúen con alelos MHC II específicos para inducir tolerancia (Treg).

In [1]:
import requests
import pandas as pd

def get_iedb_epitopes(sequence, allele="HLA-DRB1*01:01"):
    """
    Predicción de unión de péptidos a MHC II usando la API de IEDB.
    Basado en el enfoque de epítopos para Der p 23 y Der p 1.
    """
    url = "https://tools.iedb.org/tools_api/mhcii/"
    payload = {
        "method": "recommended",
        "sequence_text": sequence,
        "allele": allele,
        "length": "15"
    }
    response = requests.post(url, data=payload)
    # Procesar respuesta (formato texto plano/CSV de IEDB)
    print("Análisis de Epítopos Completado para Alelo:", allele)
    return response.text

# Ejemplo: Secuencia parcial de Der p 23 mencionada en el reporte
der_p_23_seq = "MANDDDPTTVHPTTTEQPDDKCPSRFGYFADPKDPHKFEKY"
# epitopes = get_iedb_epitopes(der_p_23_seq)

# 3. Módulo CRISPR-Design: Modulación de la Memoria IgE
Siguiendo la Fase II, este módulo simula el diseño de un sgRNA dirigido a la región de cambio   $S\epsilon$ para bloquear el cambio de isotipo a IgE.

In [4]:
from Bio.Seq import Seq

def design_sgrna_target(target_region_seq):
    """
    Identifica sitios PAM (NGG) y genera candidatos a sgRNA (20nt).
    Objetivo: Región S-epsilon del locus IgH.
    """
    targets = []
    for i in range(len(target_region_seq) - 23):
        subseq = target_region_seq[i:i+23]
        if subseq.endswith("GG"): # PAM site para Cas9
            guide = subseq[:20]
            targets.append(guide)
    return targets

# Secuencia ejemplo de la región de cambio S-epsilon (Placeholder)
s_epsilon_region = "GGGCTAGCTAGCTAGCTAGCTGATCGATCGATCGCGATCGATCGATCGATCGATCGGG"
guides = design_sgrna_target(s_epsilon_region)
print(f"Detectados {len(guides)} candidatos a sgRNA para bloqueo de IgE.")

Detectados 1 candidatos a sgRNA para bloqueo de IgE.


# 4. Módulo de Medicina de Precisión: Predictor de Eficacia (AIT-XGBoost)  
Este es el núcleo de la Fase IV. Implementa el modelo matemático del reporte para predecir si un paciente será "Respondedor" tras 15 semanas de inmunoterapia.

In [5]:
import numpy as np
import xgboost as xgb

def predict_ait_efficacy(v1_csms, delta_der_f1_sige, delta_der_p23_sigg4):
    """
    Implementa la fórmula de regresión logística binaria del reporte:
    y = -6.749 + 1.823(V1 CSMS) + 0.068(Δ15w Der f 1 sIgE) + 0.003(Δ15w Der p 23 sIgG4)
    """
    # Coeficientes del modelo clínico validado
    intercept = -6.749
    coef_csms = 1.823
    coef_sige = 0.068
    coef_sigg4 = 0.003

    logit_y = intercept + (coef_csms * v1_csms) + (coef_sige * delta_der_f1_sige) + (coef_sigg4 * delta_der_p23_sigg4)

    # Función Sigmoide para obtener probabilidad
    probability = 1 / (1 + np.exp(-logit_y))
    return probability

# Simulación de un paciente en la Hackaton
# v1_csms: Puntuación clínica basal
# delta_sige: Cambio en IgE específica a Der f 1 en semana 15
# delta_sigg4: Cambio en IgG4 específica a Der p 23 en semana 15
prob = predict_ait_efficacy(v1_csms=4.5, delta_der_f1_sige=10.2, delta_der_p23_sigg4=150.0)
print(f"Probabilidad de éxito de la Inmunoterapia (Cura/Tolerancia): {prob:.2%}")

Probabilidad de éxito de la Inmunoterapia (Cura/Tolerancia): 93.07%


# 5. Módulo SHAP: Explicabilidad para el Clínico  
Para evitar el efecto "caja negra" y permitir que el alergólogo confíe en la IA.

In [6]:
import shap

# Simulación de un dataset para entrenamiento del modelo de explicabilidad
def explain_model(model, X_sample):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    # Genera visualización de importancia de factores (IgE, IgG4, Edad, etc.)
    shap.summary_plot(shap_values, X_sample)

print("Módulo SHAP listo para interpretación de biomarcadores (sIgE/sIgG4).")

Módulo SHAP listo para interpretación de biomarcadores (sIgE/sIgG4).
